# Preparing data for TTS transformer

In [7]:
from phonemizer import phonemize
from phonemizer.separator import Separator
import re

In [58]:
def extract_phonemes(text):
    phonemes = phonemize(
        text,
        language="ru",
        backend="espeak",
        separator=Separator(phone="#", word=" | "),
        preserve_punctuation=False,
        with_stress=True,
        strip=True,
    )
    return phonemes

In [59]:
pattern = re.compile(r"\\u[0-9a-fA-F]{4}\d?")
corpus = ""
with open("../data/corpus.txt", encoding="utf-8") as f:
    for line in f.readlines():
        line = re.sub(r"[\u2020-\u203f]\d?", "", line)
        corpus += line

In [60]:
vocab = set(["<s_ph>", "</s_ph>", "</s_mel>", "<s_mel>", "<UNK_ph>", "<UNK_mel>", "<space>"])
for word in extract_phonemes(corpus).split(" | "):
    vocab.update([phoneme for phoneme in word.split("#") if phoneme.isalpha()])
vocab = list(vocab)

In [61]:
len(vocab)

69

In [64]:
phon_to_idx = {phoneme: i for i, phoneme in enumerate(vocab)}
idx_to_phon = {i: phoneme for i, phoneme in enumerate(vocab)}

def tokenize(text):
    phonemes = extract_phonemes(text)
    tokens = [phon_to_idx["<s_ph>"]]
    for word in phonemes.split(" | "):
        for phoneme in word.split("#"):
            if phoneme in phon_to_idx.keys():
                tokens.append(phon_to_idx[phoneme])
            else:
                tokens.append(phon_to_idx["<UNK_ph>"])
        tokens.append(phon_to_idx["<space>"])
    tokens.append(phon_to_idx["</s_ph>"])
    return tokens

def untokenize(tokens):
    phonemes = []
    for token in tokens:
        if token in idx_to_phon.keys():
            phonemes.append(idx_to_phon[token])
        else:
            phonemes.append("<UNK_ph>")
    return phonemes

In [65]:
text = "Это обычное тестовое предложение."
tokens = tokenize(text)
new_text = untokenize(tokens)
print(tokens)
print(new_text)

[40, 15, 31, 37, 35, 37, 59, 30, 23, 4, 37, 7, 25, 35, 18, 15, 64, 31, 37, 44, 37, 7, 25, 35, 1, 63, 50, 34, 48, 37, 20, 15, 24, 50, 7, 25, 35, 29]
['<s_ph>', 'ˈɛ', 't', 'ʌ', '<space>', 'ʌ', 'b', 'ˈy', 'tʃʲ', 'n', 'ʌ', 'j', 'ɪ', '<space>', 'tʲ', 'ˈɛ', 's', 't', 'ʌ', 'v', 'ʌ', 'j', 'ɪ', '<space>', 'p', 'rʲ', 'i', 'd', 'ɭ', 'ʌ', 'ʒ', 'ˈɛ', 'nʲ', 'i', 'j', 'ɪ', '<space>', '</s_ph>']
